In [2]:
import json
import re
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow import keras


# ============================================================
# 1. Paths
# ============================================================

BASE_PATH = Path("/home/sudhanshu_bajpai/projects/Voice_assistant")

MODEL_PATH = ("/home/sudhanshu_bajpai/projects/Voice_assistant/data/intent_model.keras")

METADATA_PATH = ("/home/sudhanshu_bajpai/projects/Voice_assistant/data/intent_metadata.json")


# ============================================================
# 2. Load model
# ============================================================

model = keras.models.load_model(
    MODEL_PATH
)


# ============================================================
# 3. Load metadata
# ============================================================

with open(
    METADATA_PATH,
    "r",
    encoding="utf-8"
) as f:

    metadata = json.load(f)


vocab = metadata["vocab"]

intents = metadata["intents"]

MAX_LEN = metadata["max_len"]


# ============================================================
# 4. Tokenizer
# ============================================================

def tokenize(text):

    return re.findall(
        r"[a-zA-Z0-9']+",
        text.lower()
    )


# ============================================================
# 5. Encode sentence
# ============================================================

def encode_sentence(sentence):

    tokens = tokenize(sentence)

    ids = []

    for token in tokens:

        ids.append(
            vocab.get(
                token,
                vocab["<UNK>"]
            )
        )

    ids = ids[:MAX_LEN]

    while len(ids) < MAX_LEN:

        ids.append(
            vocab["<PAD>"]
        )

    return np.array(
        [ids],
        dtype=np.int32
    )


# ============================================================
# 6. Prediction
# ============================================================

def predict(sentence):

    x = encode_sentence(sentence)

    probabilities = model.predict(
        x,
        verbose=0
    )[0]

    prediction = np.argmax(
        probabilities
    )

    confidence = probabilities[
        prediction
    ]

    intent = intents[
        prediction
    ]

    return intent, confidence


# ============================================================
# 7. Completely new commands
# ============================================================

test_commands = [
    # Chrome
    "get my browser running",
    "bring up the web",
    "I want to browse the internet",

    # VS Code
    "open my programming environment",
    "I want to write some code",
    "bring up my editor",

    # GPU
    "how much graphics memory do I have",
    "tell me about my graphics hardware",

    # CPU
    "which processor is installed",
    "tell me my CPU details",

    # Jupyter
    "launch my notebook environment",
    "I want to work in Jupyter",

    # Notepad
    "open a simple text editor",
    "I need somewhere to write some notes",

    # Stop
    "you can stop now",
    "shut yourself down",
]


# ============================================================
# 8. Test
# ============================================================

for command in test_commands:

    intent, confidence = predict(
        command
    )

    print(
        f"\nCommand: {command}"
    )

    print(
        f"Intent: {intent}"
    )

    print(
        f"Confidence: {confidence:.2%}"
    )


Command: get my browser running
Intent: OPEN_EDGE
Confidence: 99.97%

Command: bring up the web
Intent: OPEN_EDGE
Confidence: 98.81%

Command: I want to browse the internet
Intent: OPEN_EDGE
Confidence: 99.82%

Command: open my programming environment
Intent: OPEN_VSCODE
Confidence: 99.98%

Command: I want to write some code
Intent: OPEN_VSCODE
Confidence: 99.96%

Command: bring up my editor
Intent: OPEN_VSCODE
Confidence: 99.77%

Command: how much graphics memory do I have
Intent: CHECK_CPU
Confidence: 41.78%

Command: tell me about my graphics hardware
Intent: CHECK_GPU
Confidence: 93.64%

Command: which processor is installed
Intent: CHECK_CPU
Confidence: 98.05%

Command: tell me my CPU details
Intent: CHECK_CPU
Confidence: 98.01%

Command: launch my notebook environment
Intent: OPEN_VSCODE
Confidence: 97.50%

Command: I want to work in Jupyter
Intent: OPEN_VSCODE
Confidence: 91.86%

Command: open a simple text editor
Intent: OPEN_NOTEPAD
Confidence: 87.90%

Command: I need somewhe

In [1]:
from command_router import prepare_command

tests = [
    ("OPEN_EDGE", "edge you, make a little soft edge you"),
    ("OPEN_NOTEPAD", "open the"),
    ("OPEN_EDGE", "yeah, hello, in Microsoft Edge, search for gaming"),
    ("SEARCH_WEB", "search for gaming"),
    ("SCREENSHOT", "save this for me"),
    ("STOP_ASSISTANT", "bye bye"),
    ("STOP_ASSISTANT", "stop listening now"),
]

for intent, command in tests:
    result = prepare_command(intent, command)

    print(f"\nCommand: {command}")
    print(f"Predicted intent: {intent}")
    print(f"Status: {result['status']}")
    print(f"Message: {result['message']}")


Command: edge you, make a little soft edge you
Predicted intent: OPEN_EDGE
Status: needs_clarification
Message: I didn't hear a clear request to open Edge. Please say 'open Edge' or repeat your command.

Command: open the
Predicted intent: OPEN_NOTEPAD
Status: needs_clarification
Message: I didn't hear a clear request to open Notepad. Please repeat your command.

Command: yeah, hello, in Microsoft Edge, search for gaming
Predicted intent: OPEN_EDGE
Status: needs_clarification
Message: I heard a web-search request, but the predicted action doesn't match. Please repeat your command.

Command: search for gaming
Predicted intent: SEARCH_WEB
Status: ready
Message: None

Command: save this for me
Predicted intent: SCREENSHOT
Status: needs_clarification
Message: I didn't hear an explicit screenshot request. Please repeat the command.

Command: bye bye
Predicted intent: STOP_ASSISTANT
Status: needs_clarification
Message: Please use a clear stop command, such as 'stop listening'.

Command: sto